# Posterior-mean denoising with convMMD

This example fits a latent density from noisy one-dimensional observations and returns one posterior-mean estimate for each observation. The generated clean values are used only after fitting to compute simulation metrics.

Importance sampling is intentionally left as the public API default. Set `CONVMMD_NOTEBOOK_SMOKE=1` for the reduced release check.


In [ ]:
import os

import matplotlib.pyplot as plt

import torch

from convMMD import denoise

from convMMD.core.data import generate_1d_laplace_mixture


## 1. Latent truth and noisy observations


In [ ]:
SMOKE = os.getenv("CONVMMD_NOTEBOOK_SMOKE") == "1"

SEED = 20260830

DEVICE = "cuda" if torch.cuda.is_available() and not SMOKE else "cpu"

N_SAMPLES = 64 if SMOKE else 512

EPOCHS = 2 if SMOKE else 500

BATCH_SIZE = 32 if SMOKE else 256

latent_truth, noisy_observations, known_noise_std = generate_1d_laplace_mixture(

    n_samples=N_SAMPLES,

    noise_type="gaussian",

    noise_std_range=(0.5, 0.5),

    seed=SEED,

    device=DEVICE,

)

print(f"device={DEVICE}, smoke={SMOKE}, n={N_SAMPLES}, epochs={EPOCHS}")


## 2. Fit and denoise

Only `noisy_observations` and `known_noise_std` enter the method. Omitting `posterior_method` exercises the supported default: adaptive importance sampling.


In [ ]:
result = denoise(

    noisy_observations,

    known_noise_std,

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    warmup_epochs=0 if SMOKE else None,

    bandwidths=[0.5, 1.0, 2.0] if SMOKE else None,

    num_blocks=1 if SMOKE else 4,

    num_bins=4 if SMOKE else 16,

    hidden_features=8 if SMOKE else 32,

    num_importance_samples=512 if SMOKE else 8192,

    posterior_batch_size=32 if SMOKE else 64,

    seed=SEED + 1,

    device=DEVICE,

    verbose=not SMOKE,

)

assert result.config.posterior_method == "importance"

denoised = result.denoised


## 3. Simulation-only evaluation


In [ ]:
noisy_mse = float((noisy_observations - latent_truth).square().mean())

denoised_mse = float((denoised - latent_truth).square().mean())

print(f"Noisy MSE:    {noisy_mse:.6f}")

print(f"Denoised MSE: {denoised_mse:.6f}")

print(f"MSE change:   {denoised_mse - noisy_mse:+.6f}")

truth_cpu = latent_truth[:, 0].cpu()

noisy_cpu = noisy_observations[:, 0].cpu()

denoised_cpu = denoised[:, 0].cpu()

lower = min(truth_cpu.min(), noisy_cpu.min(), denoised_cpu.min()).item()

upper = max(truth_cpu.max(), noisy_cpu.max(), denoised_cpu.max()).item()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(truth_cpu, noisy_cpu, s=14, alpha=0.55)

axes[0].plot([lower, upper], [lower, upper], color="black", linewidth=1)

axes[0].set(xlabel="Latent truth", ylabel="Observation", title="Before denoising")

axes[1].scatter(truth_cpu, denoised_cpu, s=14, alpha=0.55, color="tab:orange")

axes[1].plot([lower, upper], [lower, upper], color="black", linewidth=1)

axes[1].set(xlabel="Latent truth", ylabel="Posterior mean", title="After denoising")

for ax in axes:

    ax.grid(alpha=0.2)

plt.tight_layout()

plt.show()


## Interpretation

The returned tensor contains posterior-mean point estimates aligned with the input rows, not independent latent draws. MSE is a simulation-only diagnostic. Importance-sampling accuracy depends on proposal overlap and sample budget; substantive work should repeat the evaluation across seeds and budgets.
